# Export Chroma Chunks to CSV

This notebook exports all chunks from Chroma collection `medical_rag` into one CSV file.

In [ ]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (CURRENT_DIR / "python-rag-service").resolve()
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = (CURRENT_DIR.parent / "python-rag-service").resolve()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError("Cannot find python-rag-service. Open notebook from Chatbot-medical root or Notebook folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import chromadb
from app.settings import settings

COLLECTION_NAME = "medical_rag"
client = chromadb.PersistentClient(path=str(settings.vector_db_dir))
collection = client.get_collection(name=COLLECTION_NAME)
payload = collection.get(include=["documents", "metadatas"])

ids = payload.get("ids", []) or []
documents = payload.get("documents", []) or []
metadatas = payload.get("metadatas", []) or []

print("Vector DB dir:", settings.vector_db_dir)
print("Collection:", COLLECTION_NAME)
print("Chunks found:", len(documents))

if not documents:
    raise RuntimeError("No chunks found in Chroma collection. Run ingest first.")

In [ ]:
import json

CORE_META_KEYS = {"chunk_id", "doc_id", "chunk_index", "chunk_length"}

def _safe_meta(i: int) -> dict:
    if i < len(metadatas) and isinstance(metadatas[i], dict):
        return dict(metadatas[i])
    return {}

def _infer_chunk_index(chunk_id: str, fallback: int = -1) -> int:
    if ":" in chunk_id:
        tail = chunk_id.rsplit(":", 1)[-1]
        if tail.isdigit():
            return int(tail)
    return fallback

rows = []
for i, text in enumerate(documents):
    text_value = str(text)
    meta = _safe_meta(i)
    raw_id = ids[i] if i < len(ids) else f"row:{i}"

    chunk_id = str(meta.get("chunk_id") or raw_id)
    doc_id = str(meta.get("doc_id") or meta.get("source") or "unknown_source")

    chunk_index = meta.get("chunk_index")
    if not isinstance(chunk_index, int):
        chunk_index = _infer_chunk_index(chunk_id, fallback=-1)

    chunk_length = meta.get("chunk_length")
    if not isinstance(chunk_length, int):
        chunk_length = len(text_value)

    extra_meta = {k: v for k, v in meta.items() if k not in CORE_META_KEYS}

    rows.append({
        "chunk_id": chunk_id,
        "doc_id": doc_id,
        "chunk_index": chunk_index,
        "chunk_length": chunk_length,
        "chunk_text": text_value,
        "metadata_json": json.dumps(extra_meta, ensure_ascii=False),
    })

print("Rows prepared:", len(rows))

In [ ]:
import csv

OUTPUT_CSV = (PROJECT_ROOT.parent / "Notebook" / "chroma_chunks_export.csv").resolve()
FIELDNAMES = [
    "chunk_id",
    "doc_id",
    "chunk_index",
    "chunk_length",
    "chunk_text",
    "metadata_json",
]

with OUTPUT_CSV.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print("CSV exported:", OUTPUT_CSV)
print("CSV size (bytes):", OUTPUT_CSV.stat().st_size)

In [ ]:
from collections import Counter

missing_chunk_id = sum(1 for r in rows if not r["chunk_id"])
missing_chunk_text = sum(1 for r in rows if not r["chunk_text"])
doc_distribution = Counter(r["doc_id"] for r in rows)

print("Row count:", len(rows))
print("Missing chunk_id:", missing_chunk_id)
print("Missing chunk_text:", missing_chunk_text)
print("Unique documents:", len(doc_distribution))
print("Top 10 doc counts:", doc_distribution.most_common(10))

print("\nSample rows (first 3):")
for sample in rows[:3]:
    print({
        "chunk_id": sample["chunk_id"],
        "doc_id": sample["doc_id"],
        "chunk_index": sample["chunk_index"],
        "chunk_length": sample["chunk_length"],
        "chunk_text_preview": sample["chunk_text"][:120],
    })